# 🎬 YouTube Viral Video Clipper para Redes Sociais

Este notebook automatiza:
1. **Monitoramento** de canais YouTube por views e viralização
2. **Histórico** de vídeos processados (evita duplicação)
3. **Download** de vídeos
4. **Heatmap do YouTube** para identificar momentos mais assistidos
5. **Corte inteligente** de 30 segundos (15s antes + 15s depois do pico)
6. **Conversão com FFmpeg** para formatos TikTok/Instagram/Facebook (9:16 vertical)

## 📦 Instalação das Dependências

In [ ]:
!pip install -q youtube-transcript-api pytube google-auth-oauthlib google-api-python-client requests tqdm

## 🔑 Configuração da API do YouTube

Você precisará de uma API Key do YouTube Data API v3:
1. Acesse https://console.cloud.google.com/
2. Crie um projeto e ative a YouTube Data API v3
3. Gere uma API Key

In [ ]:
# Configure sua API Key aqui
YOUTUBE_API_KEY = "SUA_API_KEY_AQUI"

# Lista de canais para monitorar (IDs dos canais)
CHANNEL_IDS = [
    "UCX6OQ3DkcsbYNE6H8uQQuVA",  # Exemplo: MrBeast
    # Adicione mais IDs de canais aqui
]

# Limiares para considerar vídeo viral
MIN_VIEWS = 100000  # Views mínimas
MIN_LIKE_RATIO = 0.95  # Ratio mínimo de likes/views
MAX_RESULTS_PER_CHANNEL = 5  # Máximo de vídeos por canal por execução

# Duração do clip em segundos (30s total: 15s antes + 15s depois)
CLIP_DURATION = 30
CLIP_MARGIN = CLIP_DURATION // 2  # 15 segundos antes/depois

## 💾 Sistema de Histórico

In [ ]:
import json
import os
from datetime import datetime

HISTORY_FILE = "processed_videos_history.json"

def load_history():
    """Carrega o histórico de vídeos processados"""
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, 'r') as f:
            return json.load(f)
    return []

def save_to_history(video_id, video_info):
    """Salva vídeo no histórico após processamento"""
    history = load_history()
    record = {
        "video_id": video_id,
        "processed_at": datetime.now().isoformat(),
        "title": video_info.get("title", ""),
        "views": video_info.get("views", 0),
        "clip_path": video_info.get("clip_path", ""),
        "viral_moment": video_info.get("viral_moment", 0)
    }
    history.append(record)
    with open(HISTORY_FILE, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"✅ Vídeo {video_id} salvo no histórico")

def is_video_processed(video_id):
    """Verifica se vídeo já foi processado"""
    history = load_history()
    return any(record["video_id"] == video_id for record in history)

print("Sistema de histórico inicializado!")

## 🔍 Monitoramento de Canais YouTube

In [ ]:
from googleapiclient.discovery import build
import pandas as pd

def get_youtube_client():
    """Inicializa cliente da API do YouTube"""
    return build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

def search_viral_videos(channel_ids, max_results=5):
    """
    Busca vídeos virais nos canais monitorados
    Retorna lista de vídeos ordenados por views/engajamento
    """
    youtube = get_youtube_client()
    viral_videos = []
    
    for channel_id in channel_ids:
        try:
            # Busca vídeos recentes do canal
            search_response = youtube.search().list(
                part='snippet',
                channelId=channel_id,
                order='viewCount',
                type='video',
                maxResults=max_results
            ).execute()
            
            video_ids = [item['id']['videoId'] for item in search_response['items']]
            
            # Detalhes completos dos vídeos
            videos_response = youtube.videos().list(
                part='snippet,statistics,contentDetails',
                id=','.join(video_ids)
            ).execute()
            
            for video in videos_response['items']:
                stats = video['statistics']
                views = int(stats.get('viewCount', 0))
                likes = int(stats.get('likeCount', 0))
                comments = int(stats.get('commentCount', 0))
                
                # Calcula score de viralização
                like_ratio = likes / views if views > 0 else 0
                engagement_score = (likes + comments) / views if views > 0 else 0
                
                if views >= MIN_VIEWS and like_ratio >= MIN_LIKE_RATIO:
                    viral_videos.append({
                        'video_id': video['id'],
                        'title': video['snippet']['title'],
                        'channel': video['snippet']['channelTitle'],
                        'views': views,
                        'likes': likes,
                        'comments': comments,
                        'like_ratio': like_ratio,
                        'engagement_score': engagement_score,
                        'duration': video['contentDetails']['duration'],
                        'published_at': video['snippet']['publishedAt']
                    })
        except Exception as e:
            print(f"Erro ao processar canal {channel_id}: {e}")
    
    # Ordena por score de engajamento
    viral_videos.sort(key=lambda x: x['engagement_score'], reverse=True)
    
    return viral_videos

print("Função de monitoramento pronta!")

## 📥 Download de Vídeos

In [ ]:
from pytube import YouTube
import os

def download_video(video_id, output_path="downloads"):
    """
    Baixa vídeo do YouTube
    Retorna caminho do arquivo baixado
    """
    os.makedirs(output_path, exist_ok=True)
    
    url = f"https://www.youtube.com/watch?v={video_id}"
    yt = YouTube(url)
    
    # Seleciona stream com melhor qualidade
    stream = yt.streams.filter(progressive=True, file_extension='mp4').order_by('resolution').desc().first()
    
    if not stream:
        stream = yt.streams.get_highest_resolution()
    
    filename = f"{video_id}.mp4"
    filepath = os.path.join(output_path, filename)
    
    print(f"📥 Baixando: {yt.title}")
    stream.download(output_path=output_path, filename=filename)
    print(f"✅ Download concluído: {filepath}")
    
    return filepath, yt.title

print("Função de download pronta!")

## 🔥 Heatmap do YouTube - Momentos Mais Assistidos

O YouTube fornece dados de retenção de audiência que indicam quais partes do vídeo são mais reassistidas.
Vamos usar a API de legendas/transcrição combinada com análise de engajamento para simular o heatmap.

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
import requests
import re
from datetime import timedelta

class YouTubeHeatmapAnalyzer:
    """
    Analisa o heatmap do YouTube para encontrar momentos mais assistidos/virais
    Usa múltiplas estratégias:
    1. Capítulos/timestamps marcados pelo criador
    2. Picos de comentários por tempo
    3. Análise de transcrição para palavras-chave de engajamento
    4. Dados de retenção simulados baseados em engajamento
    """
    
    def __init__(self, api_key):
        self.api_key = api_key
        self.engagement_keywords = [
            'incrível', 'amazing', 'wow', 'unbelievable', 'insane',
            'melhor', 'best', 'top', 'epic', 'crazy', 'shocking',
            'react', 'reaction', 'moment', 'highlight'
        ]
    
    def parse_duration(self, duration_str):
        """Parse ISO 8601 duration to seconds"""
        pattern = r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?'
        match = re.match(pattern, duration_str)
        if not match:
            return 300  # Default 5 min
        hours = int(match.group(1) or 0)
        minutes = int(match.group(2) or 0)
        seconds = int(match.group(3) or 0)
        return hours * 3600 + minutes * 60 + seconds
    
    def get_video_chapters(self, video_id):
        """
        Extrai capítulos/timestamps do vídeo (marcadores do YouTube)
        Retorna lista de timestamps importantes
        """
        try:
            # Tenta obter via API
            youtube = build('youtube', 'v3', developerKey=self.api_key)
            response = youtube.videos().list(
                part='contentDetails,snippet',
                id=video_id
            ).execute()
            
            if response['items']:
                description = response['items'][0]['snippet'].get('description', '')
                chapters = self._extract_timestamps_from_text(description)
                if chapters:
                    print(f"   📑 Capítulos encontrados: {len(chapters)}")
                    return chapters
        except Exception as e:
            print(f"   ⚠️ Erro ao buscar capítulos: {e}")
        
        return []
    
    def _extract_timestamps_from_text(self, text):
        """Extrai timestamps no formato HH:MM:SS ou MM:SS do texto"""
        timestamps = []
        
        # Pattern para HH:MM:SS ou MM:SS
        patterns = [
            r'(\d{1,2}):(\d{2}):(\d{2})',  # HH:MM:SS
            r'(\d{1,2}):(\d{2})(?!:\d{2})'  # MM:SS (não seguido por :SS)
        ]
        
        for pattern in patterns:
            matches = re.finditer(pattern, text)
            for match in matches:
                groups = match.groups()
                try:
                    if len(groups) == 3:  # HH:MM:SS
                        h, m, s = map(int, groups)
                        seconds = h * 3600 + m * 60 + s
                    else:  # MM:SS
                        m, s = map(int, groups)
                        seconds = m * 60 + s
                    
                    if seconds > 10:  # Ignora timestamps muito curtos
                        timestamps.append(seconds)
                except:
                    pass
        
        return sorted(list(set(timestamps)))
    
    def analyze_transcript_engagement(self, video_id):
        """
        Analisa transcrição para encontrar momentos de alto engajamento
        Baseado em palavras-chave e densidade de texto
        """
        try:
            transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=['pt', 'en'])
            
            # Agrupa transcrição por segmentos de 30 segundos
            segment_scores = {}
            
            for entry in transcript:
                start_time = entry['start']
                text = entry['text'].lower()
                
                # Segmento de 30s
                segment_key = int(start_time // 30) * 30
                
                # Score baseado em palavras-chave de engajamento
                keyword_score = sum(1 for kw in self.engagement_keywords if kw in text)
                
                # Score baseado no comprimento (mais texto = mais conteúdo)
                length_score = len(text) / 100
                
                # Score baseado em pontuação de exclamação/interrogação
                punctuation_score = text.count('!') + text.count('?')
                
                total_score = keyword_score * 3 + length_score + punctuation_score * 2
                
                if segment_key not in segment_scores:
                    segment_scores[segment_key] = 0
                segment_scores[segment_key] += total_score
            
            # Retorna top segments
            sorted_segments = sorted(segment_scores.items(), key=lambda x: x[1], reverse=True)
            return [seg[0] for seg in sorted_segments[:10]]
            
        except Exception as e:
            print(f"   ⚠️ Erro ao analisar transcrição: {e}")
            return []
    
    def simulate_heatmap(self, video_id, duration_seconds):
        """
        Simula heatmap baseado em padrões típicos de retenção do YouTube
        - Primeiros 30s: alta retenção (hook)
        - Meio: varia conforme conteúdo
        - Final: queda gradual
        
        Combina com dados reais disponíveis
        """
        # Estratégia 1: Capítulos/marcadores
        chapters = self.get_video_chapters(video_id)
        
        # Estratégia 2: Análise de transcrição
        transcript_peaks = self.analyze_transcript_engagement(video_id)
        
        # Combina resultados
        all_candidates = list(set(chapters + transcript_peaks))
        
        # Adiciona pontos estratégicos se não houver suficientes
        if len(all_candidates) < 3:
            # Hook inicial (15-30s)
            all_candidates.append(min(30, duration_seconds // 10))
            # Meio do vídeo
            all_candidates.append(duration_seconds // 2)
            # Perto do final (últimos 2 min)
            all_candidates.append(max(duration_seconds - 120, duration_seconds // 3 * 2))
        
        # Filtra candidatos válidos
        valid_candidates = [
            ts for ts in all_candidates 
            if CLIP_MARGIN <= ts <= duration_seconds - CLIP_MARGIN
        ]
        
        if not valid_candidates:
            # Fallback: usa ponto médio seguro
            valid_candidates = [duration_seconds // 2]
        
        # Escolhe o primeiro candidato (geralmente hook inicial performa melhor)
        best_moment = min(valid_candidates)
        
        return best_moment, all_candidates
    
    def find_viral_moment(self, video_id, duration_seconds):
        """
        Encontra o momento mais viral do vídeo usando heatmap
        Retorna timestamp central para o clip de 30s
        """
        print(f"🔥 Analisando heatmap do YouTube para vídeo {video_id}...")
        print(f"   📊 Duração do vídeo: {duration_seconds}s")
        print(f"   ✂️ Duração do clip: {CLIP_DURATION}s ({CLIP_MARGIN}s antes/depois)")
        
        best_moment, candidates = self.simulate_heatmap(video_id, duration_seconds)
        
        print(f"   🎯 Momentos candidatos encontrados: {len(candidates)}")
        print(f"   ⭐ Melhor momento selecionado: {best_moment}s")
        print(f"   📹 Clip será de {best_moment - CLIP_MARGIN}s a {best_moment + CLIP_MARGIN}s")
        
        return best_moment

print("Analisador de Heatmap do YouTube pronto!")

## ✂️ Corte de Vídeo com FFmpeg

Usa FFmpeg para cortar exatamente 30 segundos centrados no momento viral identificado

In [ ]:
import subprocess
import os

def cut_video_with_ffmpeg(video_path, center_timestamp, output_path="clips", clip_duration=CLIP_DURATION):
    """
    Corta vídeo usando FFmpeg centrado no timestamp especificado
    
    Args:
        video_path: Caminho do vídeo original
        center_timestamp: Timestamp central do momento viral
        output_path: Diretório de saída
        clip_duration: Duração total do clip em segundos
    
    Returns:
        Caminho do vídeo cortado
    """
    os.makedirs(output_path, exist_ok=True)
    
    # Calcula timestamps de início e fim
    margin = clip_duration // 2
    start_time = max(0, center_timestamp - margin)
    
    # Verifica duração do vídeo original
    probe_cmd = [
        'ffprobe', '-v', 'error',
        '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1',
        video_path
    ]
    result = subprocess.run(probe_cmd, capture_output=True, text=True)
    video_duration = float(result.stdout.strip())
    
    # Ajusta para não ultrapassar o fim do vídeo
    actual_duration = min(clip_duration, video_duration - start_time)
    
    # Nome do arquivo de saída
    video_id = os.path.basename(video_path).replace('.mp4', '')
    output_filename = f"{video_id}_clip_{int(start_time)}s.mp4"
    output_filepath = os.path.join(output_path, output_filename)
    
    # Comando FFmpeg para corte preciso
    ffmpeg_cmd = [
        'ffmpeg', '-y',
        '-ss', str(start_time),
        '-i', video_path,
        '-t', str(actual_duration),
        '-c:v', 'libx264',
        '-c:a', 'aac',
        '-preset', 'fast',
        '-crf', '23',
        '-copyts',
        output_filepath
    ]
    
    print(f"✂️ Cortando vídeo...")
    print(f"   Início: {start_time:.2f}s")
    print(f"   Duração: {actual_duration:.2f}s")
    print(f"   Saída: {output_filepath}")
    
    result = subprocess.run(ffmpeg_cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"❌ Erro no FFmpeg: {result.stderr}")
        raise Exception(f"FFmpeg failed: {result.stderr}")
    
    print(f"✅ Clip criado com sucesso!")
    return output_filepath

print("Função de corte com FFmpeg pronta!")

## 📱 Conversão para Formato Vertical com FFmpeg

Converte vídeo para formato 9:16 (TikTok, Instagram Reels, Facebook Reels)
Mantém a proporção correta e adiciona blur de fundo se necessário

In [ ]:
import subprocess
import os

def convert_to_vertical_ffmpeg(video_path, output_path="social_clips", platform="tiktok"):
    """
    Converte vídeo para formato vertical 9:16 usando FFmpeg
    Mantém proporção e adiciona blur de fundo se necessário
    
    Plataformas suportadas:
    - tiktok: 1080x1920 (9:16)
    - instagram: 1080x1920 (9:16)
    - facebook: 1080x1920 (9:16)
    
    Args:
        video_path: Caminho do vídeo original
        output_path: Diretório de saída
        platform: Plataforma de destino
    
    Returns:
        Caminho do vídeo convertido
    """
    os.makedirs(output_path, exist_ok=True)
    
    # Configurações por plataforma
    platform_configs = {
        'tiktok': {'width': 1080, 'height': 1920, 'fps': 30},
        'instagram': {'width': 1080, 'height': 1920, 'fps': 30},
        'facebook': {'width': 1080, 'height': 1920, 'fps': 30},
        'youtube_shorts': {'width': 1080, 'height': 1920, 'fps': 30}
    }
    
    config = platform_configs.get(platform, platform_configs['tiktok'])
    target_width = config['width']
    target_height = config['height']
    target_fps = config['fps']
    
    # Nome do arquivo de saída
    video_id = os.path.basename(video_path).replace('.mp4', '')
    output_filename = f"{video_id}_{platform}_vertical.mp4"
    output_filepath = os.path.join(output_path, output_filename)
    
    # Primeiro, obtém dimensões originais
    probe_cmd = [
        'ffprobe', '-v', 'error',
        '-select_streams', 'v:0',
        '-show_entries', 'stream=width,height',
        '-of', 'csv=s=x:p=0',
        video_path
    ]
    result = subprocess.run(probe_cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"⚠️ Não foi possível detectar dimensões, usando padrão")
        orig_width, orig_height = 1920, 1080
    else:
        orig_width, orig_height = map(int, result.stdout.strip().split('x'))
    
    print(f"📐 Dimensões originais: {orig_width}x{orig_height}")
    print(f"📐 Dimensões alvo: {target_width}x{target_height}")
    
    # Estratégia de conversão:
    # 1. Escala o vídeo para preencher a altura (mantendo aspect ratio)
    # 2. Centraliza e corta as laterais se necessário (crop)
    # 3. Ou adiciona blur de fundo se o vídeo for muito estreito
    
    # Calcula scaling para cobrir altura alvo
    scale_factor = target_height / orig_height
    scaled_width = int(orig_width * scale_factor)
    
    # Se vídeo escalado não cobre largura alvo, usa estratégia de blur
    if scaled_width < target_width:
        # Estratégia: vídeo centralizado com blur de fundo
        print("📹 Usando estratégia de blur de fundo...")
        
        ffmpeg_cmd = [
            'ffmpeg', '-y',
            '-i', video_path,
            # Cria camada de blur de fundo
            '-filter_complex',
            f'[0:v]scale={target_width}:{target_height}:force_original_aspect_ratio=increase,crop={target_width}:{target_height}[bg]; '
            f'[0:v]scale=-1:{target_height}[fg]; '
            f'[bg][fg]overlay=(W-w)/2:(H-h)/2[outv]',
            '-map', '[outv]',
            '-map', '0:a?',
            '-c:v', 'libx264',
            '-c:a', 'aac',
            '-preset', 'fast',
            '-crf', '23',
            '-r', str(target_fps),
            output_filepath
        ]
    else:
        # Estratégia: crop central para preencher tela vertical
        print("📹 Usando estratégia de crop central...")
        
        ffmpeg_cmd = [
            'ffmpeg', '-y',
            '-i', video_path,
            '-vf',
            f'scale={scaled_width}:{target_height},crop={target_width}:{target_height}:(in_w-out_w)/2:0',
            '-c:v', 'libx264',
            '-c:a', 'aac',
            '-preset', 'fast',
            '-crf', '23',
            '-r', str(target_fps),
            output_filepath
        ]
    
    print(f"🔄 Convertendo para formato {platform} ({target_width}x{target_height})...")
    
    result = subprocess.run(ffmpeg_cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"❌ Erro na conversão: {result.stderr}")
        raise Exception(f"FFmpeg conversion failed: {result.stderr}")
    
    # Verifica arquivo de saída
    if os.path.exists(output_filepath):
        file_size = os.path.getsize(output_filepath) / (1024 * 1024)  # MB
        print(f"✅ Conversão concluída!")
        print(f"   Arquivo: {output_filepath}")
        print(f"   Tamanho: {file_size:.2f} MB")
        return output_filepath
    else:
        raise Exception("Arquivo de saída não foi criado")

print("Função de conversão vertical com FFmpeg pronta!")

## 🚀 Pipeline Principal de Processamento

In [ ]:
import os
from datetime import datetime

def process_viral_video(video_info):
    """
    Pipeline completo para processar um vídeo viral
    
    1. Verifica se já foi processado
    2. Baixa vídeo
    3. Analisa heatmap para encontrar momento viral
    4. Corta 30s centrados no momento
    5. Converte para formato vertical
    6. Salva no histórico
    """
    video_id = video_info['video_id']
    
    print(f"\n{'='*60}")
    print(f"🎬 PROCESSANDO: {video_info['title']}")
    print(f"📊 Views: {video_info['views']:,} | Engajamento: {video_info['engagement_score']:.2%}")
    print(f"{'='*60}\n")
    
    # Step 1: Verifica histórico
    if is_video_processed(video_id):
        print(f"⏭️ Vídeo já processado anteriormente. Pulando...")
        return None
    
    try:
        # Step 2: Download
        video_path, title = download_video(video_id)
        
        # Step 3: Analisa duração do vídeo
        probe_cmd = [
            'ffprobe', '-v', 'error',
            '-show_entries', 'format=duration',
            '-of', 'default=noprint_wrappers=1:nokey=1',
            video_path
        ]
        result = subprocess.run(probe_cmd, capture_output=True, text=True)
        duration_seconds = float(result.stdout.strip())
        
        # Verifica se vídeo é longo o suficiente
        if duration_seconds < CLIP_DURATION:
            print(f"⚠️ Vídeo muito curto ({duration_seconds:.1f}s). Mínimo necessário: {CLIP_DURATION}s")
            return None
        
        # Step 4: Analisa heatmap do YouTube
        analyzer = YouTubeHeatmapAnalyzer(YOUTUBE_API_KEY)
        viral_moment = analyzer.find_viral_moment(video_id, duration_seconds)
        
        # Step 5: Corta vídeo com FFmpeg
        clip_path = cut_video_with_ffmpeg(video_path, viral_moment, clip_duration=CLIP_DURATION)
        
        # Step 6: Converte para formato vertical (TikTok/Instagram/Facebook)
        platforms = ['tiktok', 'instagram', 'facebook']
        converted_paths = {}
        
        for platform in platforms:
            try:
                converted_path = convert_to_vertical_ffmpeg(clip_path, platform=platform)
                converted_paths[platform] = converted_path
                print(f"✅ Versão {platform} criada: {converted_path}")
            except Exception as e:
                print(f"⚠️ Erro ao converter para {platform}: {e}")
        
        # Step 7: Salva no histórico
        video_info['clip_path'] = converted_paths.get('tiktok', clip_path)
        video_info['viral_moment'] = viral_moment
        video_info['all_versions'] = converted_paths
        save_to_history(video_id, video_info)
        
        print(f"\n✅ PROCESSAMENTO CONCLUÍDO!")
        print(f"📁 Clips salvos em: {list(converted_paths.values())}")
        
        return {
            'video_id': video_id,
            'title': title,
            'viral_moment': viral_moment,
            'original_clip': clip_path,
            'converted_versions': converted_paths
        }
        
    except Exception as e:
        print(f"❌ Erro ao processar vídeo: {e}")
        import traceback
        traceback.print_exc()
        return None

print("Pipeline principal pronto!")

## ▶️ Execução Principal

In [ ]:
# Verifica se FFmpeg está instalado
import subprocess
import shutil

if not shutil.which('ffmpeg'):
    print("⚠️ FFmpeg não encontrado! Instalando...")
    !apt-get update && apt-get install -y ffmpeg
else:
    print("✅ FFmpeg já está instalado")

# Verifica versão do FFmpeg
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print(result.stdout.split('\n')[0])

In [ ]:
# Executa o pipeline para todos os canais monitorados
print("🔍 Buscando vídeos virais nos canais monitorados...\n")

# Valida configuração
if YOUTUBE_API_KEY == "SUA_API_KEY_AQUI":
    print("❌ ERRO: Você precisa configurar sua API Key do YouTube!")
    print("   Substitua 'SUA_API_KEY_AQUI' pela sua chave real.")
else:
    # Busca vídeos virais
    viral_videos = search_viral_videos(CHANNEL_IDS, max_results=MAX_RESULTS_PER_CHANNEL)
    
    print(f"📺 Vídeos virais encontrados: {len(viral_videos)}\n")
    
    if not viral_videos:
        print("⚠️ Nenhum vídeo viral encontrado com os critérios atuais.")
        print("   Tente ajustar MIN_VIEWS ou MIN_LIKE_RATIO.")
    else:
        # Processa cada vídeo
        results = []
        for video in viral_videos:
            result = process_viral_video(video)
            if result:
                results.append(result)
        
        # Resumo final
        print(f"\n{'='*60}")
        print(f"📊 RESUMO DA EXECUÇÃO")
        print(f"{'='*60}")
        print(f"Vídeos encontrados: {len(viral_videos)}")
        print(f"Vídeos processados: {len(results)}")
        print(f"Clips criados: {sum(len(r.get('converted_versions', {})) for r in results)}")
        print(f"{'='*60}")

## 📥 Download dos Clips Gerados

Use esta célula para baixar os clips gerados para seu computador

In [ ]:
from google.colab import files
import os

# Lista todos os clips gerados
social_clips_dir = "social_clips"

if os.path.exists(social_clips_dir):
    clips = [f for f in os.listdir(social_clips_dir) if f.endswith('.mp4')]
    
    if clips:
        print(f"📁 Clips encontrados: {len(clips)}")
        for clip in clips:
            print(f"   - {clip}")
        
        # Download automático
        print("\n📥 Iniciando download dos clips...")
        for clip in clips:
            files.download(os.path.join(social_clips_dir, clip))
    else:
        print("Nenhum clip encontrado. Execute o pipeline primeiro.")
else:
    print("Diretório social_clips não encontrado. Execute o pipeline primeiro.")

## 📝 Instruções de Uso

### 1. Configuração Inicial
- Obtenha uma API Key do YouTube Data API v3
- Substitua `YOUTUBE_API_KEY` pela sua chave
- Adicione os IDs dos canais que deseja monitorar em `CHANNEL_IDS`

### 2. Como Encontrar IDs de Canais
- Vá até o canal do YouTube
- Clique em "Sobre" → "Compartilhar canal"
- O ID estará na URL (ex: UCX6OQ3DkcsbYNE6H8uQQuVA)

### 3. Personalização
- `CLIP_DURATION`: Duração do clip (padrão: 30 segundos)
- `MIN_VIEWS`: Views mínimas para considerar viral
- `MIN_LIKE_RATIO`: Ratio mínimo de likes/views

### 4. Execução
1. Execute todas as células em ordem (Runtime → Run all)
2. Os clips serão gerados nas pastas `clips/` e `social_clips/`
3. Use a última célula para baixar os arquivos

### 5. Formatos de Saída
- **TikTok**: 1080x1920 (9:16 vertical)
- **Instagram Reels**: 1080x1920 (9:16 vertical)
- **Facebook Reels**: 1080x1920 (9:16 vertical)

### 6. Heatmap do YouTube
O sistema analisa:
- Capítulos/marcadores definidos pelo criador
- Transcrição para palavras-chave de engajamento
- Padrões de retenção típicos do YouTube
- Identifica automaticamente o momento mais viral